# Sprint 2

## python
python3 -m venv .venv
source .venv/bin/activate
python3 -m pip install --upgrade pip


## llama.cpp
sudo docker run -d --gpus all -p 8080:8080 --name llm ghcr.io/ggml-org/llama.cpp:server-cuda \
  -hf unsloth/Qwen3.5-4B-GGUF:Q4_K_M --host 0.0.0.0 --port 8080 -ngl 99 -c 4096

sudo docker start llm 

curl http://localhost:8080/health

# Этап 1. Аудит и подготовка корпуса

## pip
```
pip install ftfy bs4 trafilatura
pip install ipykernel 
python -m pip install urllib3[socks]
```

## dataset
wget https://code.s3.yandex.net/ai-engineer/Итоговый_проект_спринт2/filmoskop_kb.json -O filmoskop_kb.json

In [ ]:
'''
1.0 Load primary docs
'''
import json

with open("filmoskop_kb.json", encoding="utf-8") as f:
    data = json.load(f)
docs = data["docs"]        # 181 раздел базы знаний
queries = data["queries"]  # 66 тестовых запросов с разметкой

print("Разделов:", len(docs), "| Запросов:", len(queries))

Разделов: 181 | Запросов: 66


In [2]:
'''
1.1 count junk_ratio - pre
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.05  | Junk length:  7079  | Text length:  145549


In [4]:
'''
1.2 filter HTML
'''
from bs4 import BeautifulSoup  
import trafilatura

def normalize_html(raw):
  soup = BeautifulSoup(raw, "html.parser")
  for tag in soup(["script", "style", "nav", "footer"]):
      tag.decompose()
  for node in soup.select(".cookie"):
      node.decompose()
  return soup.get_text(" ")

for t in docs:
  if t['format'] == 'html':
    txt = trafilatura.extract(t['raw'], favor_precision=False, include_comments=False) 
    if not txt:
    #  print(t['doc_id'])
     txt = normalize_html(t['raw'])
    t['raw'] = txt

# filtered = [i for i in docs if i['raw']]

# write to file
with open("tmp/docs_html.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=4)

In [ ]:
'''
1.3 encoding, normalize text
'''
import re
import unicodedata
import ftfy

def encode_text(text):
  # Если текст побит — восстанавливаем «кракозябры»
  return ftfy.fix_text(text)

def normalize_text(text):
  # Приводим юникод к единой форме NFC
  text = unicodedata.normalize("NFC", text)

  # Склеиваем слово, разорванное переносом строки
  text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

  # Неразрывные и нулевой ширины пробелы -> обычный пробел
  text = text.replace("\N{NO-BREAK SPACE}", " ")
  text = text.replace("\N{ZERO WIDTH SPACE}", " ")

  # Схлопываем подряд идущие пробелы и переносы в один пробел
  text = re.sub(r"\s+", " ", text)

  return text.strip()


for t in docs:
  if not t['raw']:
    print('>>>', t)
    continue
  if t['format'] != 'html':
    t['raw'] = encode_text(t['raw'])

In [9]:
'''
1.4 remove useless data and put period in the end of item
'''
MIN_LENGTH = 30

filtered = []
for i in docs:
  if len(i['raw']) < MIN_LENGTH \
  or 'Внимание: статья устарела' in i['raw'] \
  or 'выгружено автоматически' in i['raw'] \
  or 'Раздел временно недоступен' in i['raw'] \
  or 'Страница не найдена' in i['raw'] \
  or 'Черновик.' in i['raw'] \
  or 'черновик, не публиковать' in i['raw']:
    continue
  
  i['raw'] = i['raw'].replace("Статья была полезна? Да Нет", "").strip()
  # i['raw'] = i['raw'].replace("— Справка «Фильмоскопа» Главная → Справка →", "").strip()
  # i['raw'] = i['raw'].replace("Мы используем cookie, чтобы сервис работал лучше", "").strip()
  i['raw'] = i['raw'].replace("→", "-").strip()

  if not i['raw'].endswith("."):
    i['raw'] = f"{i['raw']}." 

  filtered.append(i)

for t in filtered:
  t['raw'] = normalize_text(t['raw'])

with open("1_corpus.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=4)

In [ ]:
'''
1.5 Check junk_ratio
'''
import re

def junk_count(text):
  if not text:
      return 0.0
  # Мусор: всё, что не буква, не цифра, не пробел и не базовый знак препинания
  junk = re.findall(r"[^\w\s.,!?;:()«»—-]", text)

  # print('>>> junk', junk)
  return len(junk) 


text_len = 0
junk_len = 0
for t in docs:
  text_len += len(t['raw'])
  junk_len += junk_count(str(t['raw']))

print('Junk ratio: ', round(junk_len/text_len, 2), ' | Junk length: ', junk_len,' | Text length: ', text_len)

Junk ratio:  0.0  | Junk length:  147  | Text length:  128446


In [11]:
'''
1.6 Check Duplicates
'''
from collections import Counter

# docs as a list
texts = [item["raw"] for item in docs]

counts = Counter(texts)
duplicates = [item for item, count in counts.items() if count > 1]
print('>>> Duplicates', duplicates)

>>> Duplicates []


# Stage 1 - Report

## Исходный датасет
Разделов: 181 | Запросов: 66 | Text length:  145549

Форматы:
- text
- html

Junk:
- Junk ratio:  0.05  
- Junk length:  7079  


## Transformations
-  HTML -> TEXT
- Encoding unification
- Text normalization
- Query normalization

- Suspicious lengths
Selected: MIN_LENGTH = 10
Решено оставить строки типа 'QX-500;2020;да' - возможно будут полезны для запросов пользователей.

- Junk symbols 
JUNK_THRESHOLD = 0.15

- Manual audit


## Final result
Junk ratio:  0.0  | Junk length:  147
No duplicates

Normalized corpus:  1_corpus.json
Normalized queries: 1_queries.json

# Этап 2. Чанкинг и эмбеддинги

## Chunking strategies

Кириллический текст обычно токенизируется по 1,8–2,5 символа на токен (по сравнению с ~4 символами на токен в английском языке), поскольку многие токенизаторы разбивают русские/кириллицевые слова на более мелкие подслова или пары байтов:
• 772 кириллических символа ≈ 310–430 токенов
• Установка лимита ниже 350 токенов приведет к тому, что большинство документов средней длины разделятся пополам, излишне фрагментируя контекст.

Рекомендуемые конфигурации по типу модели

Уровень модели встраивания Максимальный контекст Рекомендуемый размер фрагмента Перекрытие Примечания
Модели с 512 токенами (например, multilingual-e5-large, rubert-tiny2, sbert_large_nlu_ru) 512 токенов 400–450 токенов 40–50 токенов Оставляет безопасный буфер для токенов [CLS] / [SEP] и префиксов запросов (например, "passage:") без скрытого усечения.

1. 400 chars with overlap 
  - со сдвигом: chunk_size = 40
  - within text items

2. sentence based 
  - within text items 
  - max_tokens=512


Cyrillic text typically tokenizes at 1.8 to 2.5 characters per token (compared to ~4 characters per token in English) because many tokenizers split Russian/Cyrillic words into smaller subwords or byte pairs:
• 772 Cyrillic characters ≈ 310–430 tokens
• Setting a limit below 350 tokens will split most of your average-length documents in half, fragmenting context unnecessarily.

Recommended Configurations by Model Type


Embedding Model Tier	Max Context	Recommended Chunk Size	Overlap	Notes
512-token models (e.g., multilingual-e5-large, rubert-tiny2, sbert_large_nlu_ru)	512 tokens	400–450 tokens	40–50 tokens	Leaves a safe buffer for [CLS] / [SEP] tokens and query prefixes (like "passage: ") without silent truncation.


## Install
pip install tiktoken razdel
pip install sentence_transformers numpy 
pip install tqdm ipywidgets
pip install chromadb

In [ ]:
'''
2. Chunking
'''
import tiktoken
from razdel import sentenize  # сегментация русского текста на предложения

ENC = tiktoken.get_encoding("cl100k_base")

def fixed_size_chunks(text, chunk_size=512, overlap=50):
    """Нарезает текст fixed-size-чанкингом: окно из chunk_size токенов со сдвигом chunk_size − overlap."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        window = tokens[start:start + chunk_size]
        chunks.append(ENC.decode(window))
        # Сдвигаем окно: на overlap токенов меньше длины окна
        start += chunk_size - overlap
    return chunks


def sentence_chunks(text, max_tokens=512):
    """Нарезает текст sentence-based-чанкингом: набирает целые предложения, пока не упрётся в бюджет max_tokens."""
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    current = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(ENC.encode(sent))
        # Следующее предложение не влезает — закрываем чанк
        if current and current_tokens + sent_tokens > max_tokens:
            chunks.append(" ".join(current))
            current = []
            current_tokens = 0
        current.append(sent)
        current_tokens += sent_tokens
    if current:
        chunks.append(" ".join(current))
    return chunks


def describe(name, chunks):
    """Печатает сводку по списку чанков: количество, средний размер, текст на стыке соседних чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    avg = sum(sizes) / len(sizes)
    end0 = chunks[0][-24:].replace("\n", " ")
    start1 = chunks[1][:24].replace("\n", " ")
    print(f"{name}")
    print(f"  чанков: {len(chunks)}, средний размер: {avg:.0f}")
    print(f"  размеры, токенов: {sizes}")
    print(f"  чанк 1 кончается на: ...{end0!r}")
    print(f"  чанк 2 начинается с: {start1!r}...")
    print()